In [1]:
import os
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Local data path setup
path = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'

# Load via HF datasets wrapper
ds = load_dataset('csv', data_files={'train': path})['train']

# checking
df = pd.read_csv(path)
print(f"Loaded {len(df)} rows.")
df.head(2)

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 2000 rows.


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A


# **Introduction to Hugging Face transformers and datasets questions**

In [2]:
# Creating the combined text field
def concat_fields(row):
    p = str(row['prompt']) if row['prompt'] is not None else ""
    a = str(row['A']) if row['A'] is not None else ""
    row['combined_text'] = f"{p} {a}"
    return row

ds = ds.map(concat_fields)

# Q1: Exact char length at index 51
print("Length at index 51:", len(ds[51]['combined_text']))

# Set up standard tokenizer
tok = AutoTokenizer.from_pretrained("bert-base-uncased")

# Q2 & Q3: Vocab parameters
print("Vocab size:", tok.vocab_size)
print("SEP token ID:", tok.sep_token_id)

# Q4: Mass tokenization 
clean_prompts = [str(x) for x in ds['prompt']]
tokens = tok(clean_prompts, padding='max_length', truncation=True, max_length=128, return_tensors='pt')

print("Tensor shape:", tokens['input_ids'].shape)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Length at index 51: 614


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size: 30522
SEP token ID: 102
Tensor shape: torch.Size([2000, 128])


# **BERT/RoBERTa Architecture & Attention Mechanisms Questions**

In [3]:
# Q1: Head dimension math
hidden = 768
heads = 12
print("Dimension per head:", hidden // heads)

# Q2: Grab last hidden state for row 0
base_bert = AutoModel.from_pretrained("bert-base-uncased")
row0_inputs = tok(str(ds[0]['prompt']), return_tensors='pt')

with torch.no_grad():
    bert_outs = base_bert(**row0_inputs)

last_hidden = bert_outs.last_hidden_state
print("Shape of last hidden state:", last_hidden.shape)

# Q3: Sum of first 5 floats in the CLS token token vector
cls_vector = last_hidden[0, 0, :]
cls_sum = torch.sum(cls_vector[:5]).item()
print("Sum of first 5 CLS elements:", round(cls_sum, 4))

# Q4: Tracking attention matrix weights
bert_att = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
sample_text = "Light-ion fusion is a technique."
att_inputs = tok(sample_text, return_tensors='pt')

with torch.no_grad():
    att_outs = bert_att(**att_inputs)

# Finding where 'fusion' sits in the token sequence
token_list = tok.convert_ids_to_tokens(att_inputs['input_ids'][0])
fusion_idx = token_list.index("fusion")

# Extract weight: Last Layer (-1), Head 0, from CLS (0) -> fusion
target_attention = att_outs.attentions[-1]
weight = target_attention[0, 0, 0, fusion_idx].item()
print("Attention weight (CLS -> fusion):", round(weight, 4))

Dimension per head: 64


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Shape of last hidden state: torch.Size([1, 31, 768])
Sum of first 5 CLS elements: -1.2001


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Attention weight (CLS -> fusion): 0.1025


# **Context-Aware Embeddings Questions**

In [4]:
# Q1: Basic embedding similarity match
st = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

p_emb = st.encode(str(ds[0]['prompt']), convert_to_tensor=True)
b_emb = st.encode(str(ds[0]['B']), convert_to_tensor=True)
print("MiniLM Cos Sim (Row 0 vs Option B):", round(util.cos_sim(p_emb, b_emb).item(), 4))

# Setup tracking for evaluation
opts = ['A', 'B', 'C', 'D', 'E']
truths = df['answer'].tolist()
t_preds, m_preds = [], []

# Running TF-IDF Pipeline
print("Processing TF-IDF baseline...")
tfidf = TfidfVectorizer()
for _, row in df.iterrows():
    text_chunks = [str(row['prompt'])] + [str(row[o]) for o in opts]
    matrix = tfidf.fit_transform(text_chunks)
    scores = cosine_similarity(matrix[0:1], matrix[1:]).flatten()
    t_preds.append([opts[i] for i in np.argsort(scores)[::-1][:3]])

# Running Sentence-Transformers Pipeline
print("Processing MiniLM embeddings...")
for _, row in df.iterrows():
    p_vec = st.encode(str(row['prompt']), convert_to_tensor=True)
    o_vecs = st.encode([str(row[o]) for o in opts], convert_to_tensor=True)
    scores = util.cos_sim(p_vec, o_vecs).flatten().cpu().numpy()
    m_preds.append([opts[i] for i in np.argsort(scores)[::-1][:3]])

# Metric: MAP@3 evaluator
def map3_eval(preds, actual):
    out = []
    for p, a in zip(preds, actual):
        val = 0.0
        for rank, item in enumerate(p[:3]):
            if item == a:
                val = 1.0 / (rank + 1)
                break
        out.append(val)
    return np.mean(out)

# Q2 & Q3 Results
print("MiniLM MAP@3 Score:", round(map3_eval(m_preds, truths), 4))

fixed_cases = sum(1 for t, m, true in zip(t_preds, m_preds, truths) if (true not in t) and (true in m))
print("Questions caught by MiniLM but missed by TF-IDF:", fixed_cases)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

MiniLM Cos Sim (Row 0 vs Option B): 0.7658
Processing TF-IDF baseline...
Processing MiniLM embeddings...
MiniLM MAP@3 Score: 0.4231
Questions caught by MiniLM but missed by TF-IDF: 564


# **Zero-shot classification concept questions**

In [5]:
# Loading zero-shot pipeline
zshot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Test target setup (Row Index 1)
test_prompt = str(ds[1]['prompt'])
test_labels = [str(ds[1]['A']), str(ds[1]['B']), str(ds[1]['C'])]

# Q1: Multi-class setup (Softmax activation)
soft_res = zshot(test_prompt, candidate_labels=test_labels, multi_label=False)
print("Top label probability (Softmax):", round(soft_res['scores'][0], 4))

# Q2: Multi-label setup (Sigmoid activation)
sig_res = zshot(test_prompt, candidate_labels=test_labels, multi_label=True)

# Computing absolute margin differences between the distributions
diff = abs(sum(soft_res['scores']) - sum(sig_res['scores']))
print("Absolute total distribution difference:", round(diff, 4))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Top label probability (Softmax): 0.4575
Absolute total distribution difference: 0.9995


# **Generative AI Question**

In [6]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Loading the explicit Seq2Seq architecture classes manually
model_id = "google/flan-t5-small"
t5_tokenizer = AutoTokenizer.from_pretrained(model_id)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# Building our milestone prompt template
qa_prompt = f"Question: {ds[0]['prompt']}. Is the correct answer A: {ds[0]['A']} or B: {ds[0]['B']}? Answer with just the letter A or B."

# Tokenizing the input text into a numeric tensor
inputs = t5_tokenizer(qa_prompt, return_tensors="pt")

# Generating tokens directly using raw model control
output_ids = t5_model.generate(**inputs, max_new_tokens=5)

# Decoding only the new output tokens back into text
clean_output = t5_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
print("Model Output:", clean_output)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model Output: B
